# Module 2: Product Catalog Dimension Cleaning & Primary Key Deduplication
**Project:** Multi-Source Retail Analytics & Anomaly Detection Pipeline  
**Author:** Piyush Kumar Singh  
**Stack:** Python 3.12+, Pandas, NumPy  
**Input:** `products_messy.csv` (~360 raw SKU records)  
**Output:** `products_clean.csv` (Sanitized Product Dimension)

---

### Executive Summary & Business Objective
The product catalog dimension table (`products_messy.csv`) defines SKU hierarchies, categories, base pricing, and supplier mappings. In relational data warehousing, dimension tables must enforce strict primary key uniqueness. If duplicate primary keys exist in a dimension table, downstream relational joins with order fact tables cause cartesian explosions (many-to-many row multiplication), severely distorting revenue rollups.

This module focuses on:
1. Auditing the product dimension schema and identifying duplicate SKU records.
2. Resolving duplicate primary keys (`product_id`) by dropping redundant records.
3. Standardizing product category strings to uniform Title Case.
4. Exporting an immutable product dimension fact table (`products_clean.csv`).

## 1. Environment Setup & Library Initialization
Import required data manipulation libraries (`pandas` and `numpy`).

In [15]:
import numpy as np
import pandas as pd

## 2. Ingesting Product Catalog & Initial Diagnostics
Load `products_messy.csv` and inspect dimensional shape, column types, null counts, and summary statistics across all fields.

In [16]:
df=pd.read_csv('products_messy.csv')

In [17]:
df.head()

,product_id,product_name,category,brand,list_price,supplier_id
0,PROD0001,Premium Watch 1,Home & Kitchen,FitWay,NaN,SUP116
1,PROD0002,Premium Bottle 2,Home & Kitchen,DailyChoice,29539.55,SUP106
2,PROD0003,Basic Mixer 3,Grocery,UrbanLeaf,30592.76,SUP103
3,PROD0004,Classic Phone 4,Books,Nova,31072.81,SUP130
4,PROD0005,Smart Bag 5,Grocery,UrbanLeaf,1953.63,SUP125


In [18]:
df.shape

(360, 6)

In [19]:
df.dtypes

product_id          str
product_name        str
category            str
brand               str
list_price      float64
supplier_id         str
dtype: object

In [20]:
df.isnull().sum()

product_id       0
product_name     0
category         0
brand            0
list_price      20
supplier_id      0
dtype: int64

In [21]:
df.describe(include='all')

,product_id,product_name,category,brand,list_price,supplier_id
count,360,360,360,360,340.000000,360
unique,350,350,14,6,NaN,40
top,PROD0035,Smart T-Shirt 35,Books,UrbanLeaf,NaN,SUP131
freq,2,2,59,71,NaN,15
mean,NaN,NaN,NaN,NaN,25540.523000,NaN
std,NaN,NaN,NaN,NaN,14200.593647,NaN
min,NaN,NaN,NaN,NaN,959.120000,NaN
25%,NaN,NaN,NaN,NaN,12961.952500,NaN
50%,NaN,NaN,NaN,NaN,25913.395000,NaN
75%,NaN,NaN,NaN,NaN,38340.245000,NaN


## 3. Duplicate Key Audit & Primary Key Resolution
In relational dimensional modeling, each `product_id` must uniquely identify exactly one SKU entity.
- Scan for exact and partial duplicate rows.
- Inspect specific duplicate occurrences (e.g. `PROD0203`, `PROD0191`).
- Execute primary key deduplication using `df.drop_duplicates(subset='product_id')` to establish an immutable 1:1 dimension entity mapping.

In [22]:
df[df.duplicated()]

,product_id,product_name,category,brand,list_price,supplier_id
350,PROD0203,Ultra T-Shirt 203,Grocery,TechSphere,48482.54,SUP130
351,PROD0191,Premium Watch 191,Clothing,TechSphere,36273.35,SUP123
352,PROD0333,Smart Lamp 333,Sports,HomeNest,24359.93,SUP111
353,PROD0296,Premium Bag 296,Home & Kitchen,HomeNest,24419.09,SUP105
354,PROD0181,Smart Phone 181,Sports,HomeNest,49857.58,SUP127
355,PROD0035,Smart T-Shirt 35,Clothing,TechSphere,18772.57,SUP108
356,PROD0284,Ultra T-Shirt 284,Grocery,TechSphere,30134.23,SUP139
357,PROD0217,Premium Bottle 217,Electronics,FitWay,30043.14,SUP139
358,PROD0045,Premium Shoes 45,Sports,DailyChoice,19214.02,SUP129
359,PROD0224,Ultra Shoes 224,Sports,FitWay,13035.32,SUP102


In [25]:
df[df['product_id']=='PROD0203']
df[df['product_id']=='PROD0191']

,product_id,product_name,category,brand,list_price,supplier_id
190,PROD0191,Premium Watch 191,Clothing,TechSphere,36273.35,SUP123
351,PROD0191,Premium Watch 191,Clothing,TechSphere,36273.35,SUP123


In [26]:
df=df.drop_duplicates(subset='product_id')

## 4. Product Category Hierarchy Normalization
Examine unique category entries. Standardize text by stripping whitespace and enforcing Title Case formatting so categories roll up cleanly in downstream reporting.

In [28]:
df['category'].unique()

<StringArray>
[ 'Home & Kitchen',         'Grocery',           'Books',          'Beauty',
          'Sports',        'Clothing',     'Electronics', 'HOME & KITCHEN ',
       'CLOTHING ',        'GROCERY ',         'SPORTS ',    'ELECTRONICS ',
          'BOOKS ',         'BEAUTY ']
Length: 14, dtype: str

In [29]:
df['category']=df['category'].str.strip().str.title()

In [30]:
df['category'].unique()

<StringArray>
['Home & Kitchen',        'Grocery',          'Books',         'Beauty',
         'Sports',       'Clothing',    'Electronics']
Length: 7, dtype: str

## 5. Supplier Mapping Validation & Production Export
Audit supplier identifiers (`supplier_id`) to ensure categorical integrity, then persist the sanitized product dimension to `products_clean.csv`.

In [34]:
df['supplier_id'].unique()

<StringArray>
['SUP116', 'SUP106', 'SUP103', 'SUP130', 'SUP125', 'SUP121', 'SUP117',
 'SUP120', 'SUP133', 'SUP107', 'SUP119', 'SUP109', 'SUP104', 'SUP127',
 'SUP128', 'SUP134', 'SUP100', 'SUP122', 'SUP132', 'SUP137', 'SUP131',
 'SUP136', 'SUP112', 'SUP123', 'SUP114', 'SUP135', 'SUP108', 'SUP105',
 'SUP118', 'SUP110', 'SUP129', 'SUP111', 'SUP126', 'SUP138', 'SUP115',
 'SUP139', 'SUP124', 'SUP101', 'SUP113', 'SUP102']
Length: 40, dtype: str

In [35]:
df.to_csv('products_clean.csv',index=False)